In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)


In [ ]:
df = pd.read_csv(
    "climate-deforestation-merged.csv"
)

print("Kolumny wylesianie:")
print(df.columns)


Kolumny wylesianie:
Index(['Year', 'Month', 'State', 'Deforestation_ha', 'CO2_Emissions',
       'Total_Precipitation', 'Atmospheric_Pressure_Station',
       'Global_Radiation', 'Air_Temperature', 'Max_Air_Temperature',
       'Min_Air_Temperature', 'Relative_Humidity', 'Max_Relative_Humidity',
       'Min_Relative_Humidity', 'Hourly_Wind_Speed', 'Max_Wind_Gust'],
      dtype='object')


In [ ]:

features = [
    "Deforestation_ha",
    "CO2_Emissions"
]

target = "Air_Temperature"

X = df[features]

y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = LinearRegression()

model.fit(
    X_train,
    y_train
)

pred = model.predict(
    X_test
)

print("R² =", r2_score(y_test, pred))

R² = 0.01856674509957601


In [ ]:
print(
    df[["Deforestation_ha",
        "Air_Temperature"]]
    .corr()
)

                  Deforestation_ha  Air_Temperature
Deforestation_ha          1.000000         0.128436
Air_Temperature           0.128436         1.000000


In [ ]:
X = df[[
    "Deforestation_ha",
    "CO2_Emissions"
]]


y = df["Total_Precipitation"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = LinearRegression()

model.fit(
    X_train,
    y_train
)

pred = model.predict(X_test)

r2 = r2_score(y_test, pred)

mae = mean_absolute_error(y_test, pred)

rmse = mean_squared_error(
    y_test,
    pred
) ** 0.5

print("R² =", r2)
print("MAE =", mae)
print("RMSE =", rmse)

coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

print("\nWpływ zmiennych:")
print(coef_df)

print("\nIntercept:")
print(model.intercept_)

R² = 0.0005354374954930075
MAE = 113.06661104596333
RMSE = 164.2203894762712

Wpływ zmiennych:
            Feature  Coefficient
0  Deforestation_ha    -0.003012
1     CO2_Emissions     0.000004

Intercept:
154.9268739830551


In [ ]:

df = df.sort_values(
    ["State", "Year"]
)

df["Deforestation_lag1"] = (
    df.groupby("State")["Deforestation_ha"]
      .shift(1)
)

df["Deforestation_lag2"] = (
    df.groupby("State")["Deforestation_ha"]
      .shift(2)
)

df["Deforestation_lag3"] = (
    df.groupby("State")["Deforestation_ha"]
      .shift(3)
)

df_model = df.dropna(
    subset=[
        "Deforestation_lag1",
        "Deforestation_lag2",
        "Deforestation_lag3"
    ]
)


X = df_model[[
    "Deforestation_lag1",
    "Deforestation_lag2",
    "Deforestation_lag3",
    "CO2_Emissions"
]]


y = df_model["Total_Precipitation"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = LinearRegression()

model.fit(
    X_train,
    y_train
)

pred = model.predict(
    X_test
)


print("R² =", r2_score(y_test, pred))

print(
    "MAE =",
    mean_absolute_error(
        y_test,
        pred
    )
)

print(
    "RMSE =",
    mean_squared_error(
        y_test,
        pred
    ) ** 0.5
)

coef_df = pd.DataFrame({

    "Feature": X.columns,
    "Coefficient": model.coef_

})

print("\nWpływ zmiennych:")
print(coef_df)

print("\nIntercept:")
print(model.intercept_)

R² = 0.014527709925966636
MAE = 104.53742177464102
RMSE = 158.47503752371153

Wpływ zmiennych:
              Feature  Coefficient
0  Deforestation_lag1    -0.001390
1  Deforestation_lag2    -0.002787
2  Deforestation_lag3     0.003215
3       CO2_Emissions     0.000002

Intercept:
148.50522897084326


In [ ]:
amazon_states = [
    "AMAZONAS",
    "PARA",
    "ACRE",
    "RONDONIA",
    "RORAIMA",
    "AMAPA",
    "TOCANTINS",
    "MARANHAO",
    "MATO GROSSO"
]

In [ ]:


amazon_df = df.sort_values(
    ["State", "Year"]
)

for lag in [1, 2, 3]:

    amazon_df[f"Deforestation_lag{lag}"] = (
        amazon_df.groupby("State")
        ["Deforestation_ha"]
        .shift(lag)
    )

amazon_df = amazon_df.dropna(
    subset=[
        "Deforestation_lag1",
        "Deforestation_lag2",
        "Deforestation_lag3"
    ]
)

X = amazon_df[[
    "Deforestation_lag1",
    "Deforestation_lag2",
    "Deforestation_lag3",
    "CO2_Emissions"
]]

y = amazon_df["Total_Precipitation"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


model = LinearRegression()

model.fit(
    X_train,
    y_train
)

pred = model.predict(
    X_test
)

print("R² =", r2_score(y_test, pred))

print(
    "MAE =",
    mean_absolute_error(
        y_test,
        pred
    )
)

print(
    "RMSE =",
    mean_squared_error(
        y_test,
        pred
    ) ** 0.5
)

coef_df = pd.DataFrame({

    "Feature": X.columns,
    "Coefficient": model.coef_

})

print(coef_df)

R² = 0.014527709925966636
MAE = 104.53742177464102
RMSE = 158.47503752371153
              Feature  Coefficient
0  Deforestation_lag1    -0.001390
1  Deforestation_lag2    -0.002787
2  Deforestation_lag3     0.003215
3       CO2_Emissions     0.000002
